In [ ]:
# Walmart Sales ML - Quick Start Guide

Welcome to the Walmart Sales ML project! 🎯

This notebook will walk you through:
1. Loading the data
2. Feature engineering
3. Time-based train/test split
4. Training a baseline model
5. Evaluating performance

**Prerequisites**: Make sure you have the dataset at `data/raw/walmart_store_sales.csv`

If you don't have the dataset yet, you can create a synthetic sample for testing!

## Setup & Imports

In [ ]:
# Add the src directory to the path
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

# Import project modules
from src.config import (
    RAW_DATA_FILE, 
    TARGET_STORE,
    TRAIN_START_DATE, 
    TRAIN_END_DATE,
    TEST_START_DATE, 
    TEST_END_DATE
)
from src.data.load_data import load_raw_data, get_data_summary
from src.features.build_features import build_features
from src.data.split_time import get_train_test_split
from src.utils.metrics import calculate_metrics, print_metrics

# Standard libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

print("✓ All imports successful!")

## 1️⃣ Load Data

We'll load the Walmart sales data for Store #1.

In [ ]:
# Load data for Store 1
df = load_raw_data(RAW_DATA_FILE, store_id=TARGET_STORE)

# Display summary
get_data_summary(df)

# Show first few rows
df.head()

## 2️⃣ Feature Engineering

Extract time-based features from the Date column.

In [ ]:
# Build features (date features: year, month, week_of_year)
df_features = build_features(df, include_date_features=True)

print(f"Original columns: {list(df.columns)}")
print(f"New columns: {list(df_features.columns)}")
print(f"\nNew features added: {[col for col in df_features.columns if col not in df.columns]}")

df_features.head()

## 3️⃣ Train/Test Split (Time-based)

⚠️ **Important**: For time-series data, we must split chronologically, not randomly!
- **Train**: 2010-2011 (earlier data)
- **Test**: 2012 (future data)

In [ ]:
# Time-based split
X_train, X_test, y_train, y_test = get_train_test_split(
    df_features,
    train_start=TRAIN_START_DATE,
    train_end=TRAIN_END_DATE,
    test_start=TEST_START_DATE,
    test_end=TEST_END_DATE
)

print("\nFeatures being used:")
print(list(X_train.columns))

## 4️⃣ Train Baseline Model

Let's train a simple Linear Regression model as our baseline.

In [ ]:
# Train Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

print("✓ Model trained successfully!")

## 5️⃣ Evaluate Performance

Let's see how well our model performs on both training and test sets.

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate metrics
train_metrics = calculate_metrics(y_train, y_pred_train)
test_metrics = calculate_metrics(y_test, y_pred_test)

# Print results
print_metrics(train_metrics, "Training Set")
print_metrics(test_metrics, "Test Set")

## 📊 Visualize Predictions (Optional)

Let's compare actual vs predicted sales.

In [ ]:
import matplotlib.pyplot as plt

# Create prediction comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Actual vs Predicted (scatter)
ax1.scatter(y_test, y_pred_test, alpha=0.5)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_xlabel('Actual Sales ($)')
ax1.set_ylabel('Predicted Sales ($)')
ax1.set_title('Actual vs Predicted Sales (Test Set)')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction errors
errors = y_test - y_pred_test
ax2.hist(errors, bins=30, edgecolor='black')
ax2.axvline(x=0, color='r', linestyle='--', lw=2)
ax2.set_xlabel('Prediction Error ($)')
ax2.set_ylabel('Frequency')
ax2.set_title('Prediction Error Distribution')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Prediction Error: ${errors.mean():,.2f}")
print(f"Std Prediction Error: ${errors.std():,.2f}")

## 🎯 Summary

Congratulations! You've completed the basic ML pipeline:

✅ Loaded and explored the data  
✅ Engineered time-based features  
✅ Split data properly for time-series  
✅ Trained a baseline model  
✅ Evaluated performance  

### Next Steps

1. **Try different models**: Random Forest, Gradient Boosting, etc.
2. **Add more features**: Lag features, rolling averages
3. **Tune hyperparameters**: Grid search, cross-validation
4. **Analyze errors**: Which predictions are far off? Why?
5. **Expand to other stores**: Can the model generalize?

Check out the other notebooks:
- `01_eda.ipynb` - Exploratory Data Analysis
- `02_model_tests.ipynb` - Try different models

Happy learning! 🚀